In [5]:
import os

import numpy as np
import torch
from scipy.io import wavfile
from transformers import VitsModel, VitsTokenizer

MODEL_PATH = "./mms-rus-finetuned-final"  # Path to your saved fine-tuned model
OUTPUT_DIR = "./generated_audio"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading model and tokenizer...")
tokenizer = VitsTokenizer.from_pretrained(MODEL_PATH)
model = VitsModel.from_pretrained(MODEL_PATH).to(device)
model.eval() # Set to evaluation mode

text_to_speak = "Здравствуйте! Это тест синтезированной речи на основе моих данных. Надеюсь, голос звучит естественно и четко."

inputs = tokenizer(text_to_speak, return_tensors="pt").to(device)

with torch.no_grad():
    # VITS-specific generation parameters:
    # - noise_scale: Controls voice variation. Lower (0.667) = more stable/deterministic. Higher (0.8-1.0) = more natural but risk of artifacts in fine-tuned models.
    # - noise_scale_w: Controls duration variation. Lower (0.8) = more stable pacing.
    # - length_scale: Controls speaking speed. 1.0 = normal, 1.2 = slower, 0.8 = faster.
    output = model.generate(
        **inputs,
        noise_scale=0.667,      # Start here. Increase to 0.8 if it sounds too monotone.
        noise_scale_w=0.8,      # Keep around 0.8 for stable durations.
        length_scale=1.0        # Adjust to 1.1 or 1.2 if the fine-tuned voice speaks too fast.
    )

# The output waveform is a tensor of shape (1, 1, seq_len). We squeeze it to (seq_len,)
audio_tensor = output.waveform.squeeze().cpu().numpy()

# VITS/MMS outputs audio in the range [-1, 1]. We need to scale it to 16-bit PCM for standard WAV files.
audio_int16 = (audio_tensor * 32767).astype(np.int16)

sample_rate = 16000
output_file = os.path.join(OUTPUT_DIR, "output.wav")
wavfile.write(output_file, sample_rate, audio_int16)


NameError: name 'os' is not defined